In [3]:
# load imports and data

import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw_adoptions.csv")

print(df.shape)
df.head()

(173775, 12)


,animal_id,date_of_birth,datetime,monthyear,outcome_type,outcome_subtype,animal_type,sex_upon_outcome,age_upon_outcome,breed,color,name
0,A668305,2012-12-01,2013-12-02T00:00:00-05:00,12-2013,Transfer,Partner,Other,Unknown,1 year,Turtle Mix,Brown/Yellow,NaN
1,A673335,2012-02-22,2014-02-22T00:00:00-05:00,02-2014,Euthanasia,Suffering,Other,Unknown,2 years,Raccoon,Black/Gray,NaN
2,A675999,2013-04-03,2014-04-07T00:00:00-05:00,04-2014,Transfer,Partner,Other,Unknown,1 year,Turtle Mix,Green,NaN
3,A679066,2014-04-16,2014-05-16T00:00:00-05:00,05-2014,NaN,NaN,Other,Unknown,4 weeks,Rabbit Sh,Brown,NaN
4,A680855,2014-05-25,2014-06-10T00:00:00-05:00,06-2014,Transfer,Partner,Bird,Unknown,2 weeks,Duck,Yellow/Black,NaN


In [4]:
# drop unnecessary cols

df = df.drop(columns=['monthyear', 'outcome_subtype'])

print(df.shape)
print(df.columns.tolist())

(173775, 10)
['animal_id', 'date_of_birth', 'datetime', 'outcome_type', 'animal_type', 'sex_upon_outcome', 'age_upon_outcome', 'breed', 'color', 'name']


In [5]:
# create binary target var
# 0 = not adopted: transfer, rto, euth, etc
# 1 = adopted: adoption, rto-adopt

# drop rows where outcome_type is missing
df = df.dropna(subset=['outcome_type'])

# create "adopted" binary var, group "adoption" and "rto-adopt" into "adopted"; all other outcomes are "not adopted"
df['adopted'] = df['outcome_type'].isin(['Adoption', 'Rto-Adopt']).astype(int)

# check value counts and % for new var
print(df['adopted'].value_counts())
print(df['adopted'].value_counts(normalize=True).round(3) * 100)

adopted
0    87890
1    85839
Name: count, dtype: int64
adopted
0    50.6
1    49.4
Name: proportion, dtype: float64


In [6]:
# feature engineering: sex_upon_outcome
# split sex and fixed status into separate features

# create "is_fixed" binary feature
df['is_fixed'] = df['sex_upon_outcome'].isin(['Neutered Male', 'Spayed Female']).astype(int)

# extract sex
def extract_sex(value):
    if pd.isna(value):
        return 'Unknown'
    if 'Male' in value:
        return 'Male'
    elif 'Female' in value:
        return 'Female'
    else:
        return 'Unknown'

# create "sex" feature using extracted_sex
df['sex'] = df['sex_upon_outcome'].apply(extract_sex)

# check value counts for new features
print(df['is_fixed'].value_counts())
print(df['is_fixed'].map({1: 'Fixed', 0: 'Not Fixed'}).value_counts(normalize=True).round(3) * 100)
print(df['sex'].value_counts())
print(df['sex'].value_counts(normalize=True).round(3) * 100)

is_fixed
1    116176
0     57553
Name: count, dtype: int64
is_fixed
Fixed        66.9
Not Fixed    33.1
Name: proportion, dtype: float64
sex
Male       83178
Female     77055
Unknown    13496
Name: count, dtype: int64
sex
Male       47.9
Female     44.4
Unknown     7.8
Name: proportion, dtype: float64


In [7]:
# drop unnecessary sex_upon_outcome col

df = df.drop(columns=['sex_upon_outcome'])

print(df.columns.tolist())

['animal_id', 'date_of_birth', 'datetime', 'outcome_type', 'animal_type', 'age_upon_outcome', 'breed', 'color', 'name', 'adopted', 'is_fixed', 'sex']


In [8]:
# feature engineering: has_name

# create binary "has_name" feature
df['has_name'] = df['name'].notna().astype(int)

# check new feature
print(df['has_name'].value_counts())

# drop original name col since what's needed is extracted
df = df.drop(columns=['name'])

print(df.columns.tolist())

has_name
1    123962
0     49767
Name: count, dtype: int64
['animal_id', 'date_of_birth', 'datetime', 'outcome_type', 'animal_type', 'age_upon_outcome', 'breed', 'color', 'adopted', 'is_fixed', 'sex', 'has_name']


In [9]:
# convert age str into numeric value in days for ML model

def convert_age_to_days(age_str):
    if pd.isna(age_str):
        return None
    
    parts = age_str.lower().split()
    if len(parts) < 2:
        return None
        
    value = int(parts[0])
    unit = parts[1]
    
    if 'day' in unit:
        return value
    elif 'week' in unit:
        return value * 7
    elif 'month' in unit:
        return value * 30
    elif 'year' in unit:
        return value * 365
    else:
        return None

# apply conversion fn to every row and store as a new col
df['age_in_days'] = df['age_upon_outcome'].apply(convert_age_to_days)

# remove negative ages, treat as missing rather than valid data
df.loc[df['age_in_days'] < 0, 'age_in_days'] = None

# drop original str col now that there is clean numeric version
df = df.drop(columns=['age_upon_outcome'])

# check new col, confirm how many missing values remain
print(df['age_in_days'].describe())
print(f"Missing ages: {df['age_in_days'].isna().sum()}")

count    173710.000000
mean        723.309050
std        1021.741069
min           0.000000
25%          60.000000
50%         365.000000
75%         730.000000
max       10950.000000
Name: age_in_days, dtype: float64
Missing ages: 19


In [10]:
# check all missing vals in df

print(df.isnull().sum())

animal_id         0
date_of_birth     0
datetime          0
outcome_type      0
animal_type       0
breed             0
color             0
adopted           0
is_fixed          0
sex               0
has_name          0
age_in_days      19
dtype: int64


In [11]:
# drop the 19 rows with missing ages, too few to be significant
df = df.dropna(subset=['age_in_days'])

# confirm no missing values remain
print(df.isnull().sum())
print(f"Total rows remaining: {len(df)}")

animal_id        0
date_of_birth    0
datetime         0
outcome_type     0
animal_type      0
breed            0
color            0
adopted          0
is_fixed         0
sex              0
has_name         0
age_in_days      0
dtype: int64
Total rows remaining: 173710


In [13]:
# convert dtm cols from strs to dtms

df['date_of_birth'] = pd.to_datetime(df['date_of_birth'])
df['datetime'] = pd.to_datetime(df['datetime'])

# extract individual features from dtm
df['outcome_year'] = df['datetime'].dt.year
df['outcome_month'] = df['datetime'].dt.month
df['outcome_weekday'] = df['datetime'].dt.dayofweek

# check results
print(df[['datetime', 'outcome_year', 'outcome_month', 'outcome_weekday']].head())
print()
print(df.dtypes)

ValueError: time data "2013-10-01T09:31:00" doesn't match format "%Y-%m-%dT%H:%M:%S%z", at position 745. You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.

In [14]:
# investigate str to dtm conversion error
# dtm col has vals that can't be parsed?

bad_dates = pd.to_datetime(df['datetime'], errors='coerce')
print(df[bad_dates.isna()]['datetime'].value_counts())

datetime
2019-04-26T00:00:00    21
2014-10-20T09:00:00    16
2016-10-29T09:00:00    15
2017-07-03T15:05:00    15
2014-07-16T09:00:00    14
                       ..
2017-03-01T07:42:00     1
2017-02-28T19:05:00     1
2017-02-28T18:49:00     1
2017-02-28T18:30:00     1
2018-08-26T16:49:00     1
Name: count, Length: 145059, dtype: int64


In [17]:
# dtms look valid
# error due to Python "datetime" keyword?

# rename to avoid potential conflict with Python's dtm module
df = df.rename(columns={'datetime': 'outcome_datetime'})

# convert str to dtm
df['date_of_birth'] = pd.to_datetime(df['date_of_birth'])
df['outcome_datetime'] = pd.to_datetime(df['outcome_datetime'])

# extract individual dtm features
df['outcome_year'] = df['outcome_datetime'].dt.year
df['outcome_month'] = df['outcome_datetime'].dt.month
df['outcome_weekday'] = df['outcome_datetime'].dt.dayofweek

# check results
print(df[['outcome_datetime', 'outcome_year', 'outcome_month', 'outcome_weekday']].head())
print()
print(df.dtypes)

ValueError: time data "2013-10-01T09:31:00" doesn't match format "%Y-%m-%dT%H:%M:%S%z", at position 745. You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.

In [21]:
# error due to dates in ISO8601 format?

df['outcome_datetime'] = pd.to_datetime(df['outcome_datetime'], format='ISO8601')

In [22]:
# no error above means adding format helped, run the rest of the dtm conversion

# extract individual dtm features
df['outcome_year'] = df['outcome_datetime'].dt.year
df['outcome_month'] = df['outcome_datetime'].dt.month
df['outcome_weekday'] = df['outcome_datetime'].dt.dayofweek

# check results
print(df[['outcome_datetime', 'outcome_year', 'outcome_month', 'outcome_weekday']].head())
print()
print(df.dtypes)

           outcome_datetime  outcome_year  outcome_month  outcome_weekday
0 2013-12-02 00:00:00-05:00          2013             12                0
1 2014-02-22 00:00:00-05:00          2014              2                5
2 2014-04-07 00:00:00-05:00          2014              4                0
4 2014-06-10 00:00:00-05:00          2014              6                1
5 2014-06-10 00:00:00-05:00          2014              6                1

animal_id                              object
date_of_birth                  datetime64[ns]
outcome_datetime    datetime64[ns, UTC-05:00]
outcome_type                           object
animal_type                            object
breed                                  object
color                                  object
adopted                                 int64
is_fixed                                int64
sex                                    object
has_name                                int64
age_in_days                           float64
out

In [23]:
# outcome_datetime is UTC-05:00 offset, date_of_birth is not
# remove timezone from outcome_datetime for consistency

df['outcome_datetime'] = df['outcome_datetime'].dt.tz_localize(None)

# check results
print(df['outcome_datetime'].dtype)
print(df['outcome_datetime'].head())

datetime64[ns]
0   2013-12-02
1   2014-02-22
2   2014-04-07
4   2014-06-10
5   2014-06-10
Name: outcome_datetime, dtype: datetime64[ns]


In [24]:
# drop unnecessary dtm cols

df = df.drop(columns=['date_of_birth', 'outcome_datetime'])

print(df.columns.tolist())
print(df.shape)

['animal_id', 'outcome_type', 'animal_type', 'breed', 'color', 'adopted', 'is_fixed', 'sex', 'has_name', 'age_in_days', 'outcome_year', 'outcome_month', 'outcome_weekday']
(173710, 13)


In [25]:
# explore breed / color

print(f"Unique colors: {df['color'].nunique()}")
print(df['color'].value_counts().head(10))

Unique colors: 663
color
Black/White          17991
Black                14677
Brown Tabby          10701
Brown                 7143
White                 6127
Brown/White           5570
Brown Tabby/White     5465
Orange Tabby          5149
Tan/White             4971
White/Black           4818
Name: count, dtype: int64


In [26]:
# 663 unique colors is too many, group into primary color and add binary for multi-color

# extract primary color (everything before slash)
df['primary_color'] = df['color'].str.split('/').str[0].str.strip()

# create binary for multiple colors
df['is_multicolor'] = df['color'].str.contains('/').astype(int)

# check results
print(f"Unique primary colors: {df['primary_color'].nunique()}")
print()
print(df['primary_color'].value_counts().head(10))
print()
print(df['is_multicolor'].map({1: 'Multicolor', 0: 'Single Color'}).value_counts(normalize=True).round(3) * 100)

Unique primary colors: 59

primary_color
Black            42117
White            22082
Brown            17492
Brown Tabby      16350
Tan              11359
Blue              8247
Orange Tabby      7747
Brown Brindle     4513
Red               4171
Tricolor          4149
Name: count, dtype: int64

is_multicolor
Multicolor      53.0
Single Color    47.0
Name: proportion, dtype: float64


In [27]:
# drop unnecessary color col

df = df.drop(columns=['color'])

print(df.columns.tolist())

['animal_id', 'outcome_type', 'animal_type', 'breed', 'adopted', 'is_fixed', 'sex', 'has_name', 'age_in_days', 'outcome_year', 'outcome_month', 'outcome_weekday', 'primary_color', 'is_multicolor']


In [28]:
# feature engineering: breed

# binary feature for is_mix
df['is_mix'] = df['breed'].str.contains('Mix', case=False).astype(int)

# extract primary breed (everything before slash)
df['primary_breed'] = df['breed'].str.split('/').str[0].str.strip()
df['primary_breed'] = df['primary_breed'].str.replace(' Mix', '', case=False).str.strip()

# check results
print(f"Unique primary breeds: {df['primary_breed'].nunique()}")
print()
print(df['primary_breed'].value_counts().head(10))
print()
print(df['is_mix'].map({1: 'Mix', 0: 'Not Mix'}).value_counts(normalize=True).round(3) * 100)

Unique primary breeds: 386

primary_breed
Domestic Shorthair       57890
Pit Bull                 14479
Labrador Retriever       13536
Chihuahua Shorthair      10114
German Shepherd           6988
Domestic Medium Hair      5485
Bat                       4118
Australian Cattle Dog     3153
Domestic Longhair         2493
Siberian Husky            2117
Name: count, dtype: int64

is_mix
Mix        60.8
Not Mix    39.2
Name: proportion, dtype: float64


In [29]:
# 386 unique breeds is still too many
# keep top 20 breeds and group the rest as "other"

top_breeds = df['primary_breed'].value_counts().head(20).index
df['primary_breed'] = df['primary_breed'].apply(lambda x: x if x in top_breeds else 'Other')

# check results
print(f"Unique primary breeds after grouping: {df['primary_breed'].nunique()}")
print()
print(df['primary_breed'].value_counts())

Unique primary breeds after grouping: 21

primary_breed
Domestic Shorthair       57890
Other                    37420
Pit Bull                 14479
Labrador Retriever       13536
Chihuahua Shorthair      10114
German Shepherd           6988
Domestic Medium Hair      5485
Bat                       4118
Australian Cattle Dog     3153
Domestic Longhair         2493
Siberian Husky            2117
Dachshund                 2109
Siamese                   2050
Boxer                     1885
Border Collie             1789
Great Pyrenees            1563
Miniature Poodle          1389
Raccoon                   1365
Australian Shepherd       1286
Catahoula                 1260
Beagle                    1221
Name: count, dtype: int64


In [30]:
# drop unncessary breed col

df = df.drop(columns=['breed'])

print(df.columns.tolist())
print(df.shape)

['animal_id', 'outcome_type', 'animal_type', 'adopted', 'is_fixed', 'sex', 'has_name', 'age_in_days', 'outcome_year', 'outcome_month', 'outcome_weekday', 'primary_color', 'is_multicolor', 'is_mix', 'primary_breed']
(173710, 15)


In [31]:
# review all cols

print(df.dtypes)
print()
print(df.head())

animal_id           object
outcome_type        object
animal_type         object
adopted              int64
is_fixed             int64
sex                 object
has_name             int64
age_in_days        float64
outcome_year         int32
outcome_month        int32
outcome_weekday      int32
primary_color       object
is_multicolor        int64
is_mix               int64
primary_breed       object
dtype: object

  animal_id outcome_type animal_type  adopted  is_fixed      sex  has_name  \
0   A668305     Transfer       Other        0         0  Unknown         0   
1   A673335   Euthanasia       Other        0         0  Unknown         0   
2   A675999     Transfer       Other        0         0  Unknown         0   
4   A680855     Transfer        Bird        0         0  Unknown         0   
5   A680857     Transfer        Bird        0         0  Unknown         0   

   age_in_days  outcome_year  outcome_month  outcome_weekday primary_color  \
0        365.0          2013     

In [32]:
# animal_id and outcome_type need to be dropped before modeling
# animal_id is an identifier, not predictive
# outcome_type is the original str that we converted to "adopted" binary

df = df.drop(columns=['animal_id', 'outcome_type'])

In [34]:
# categorical cols need to be encoded
# animal_type, sex, primary_color, primary_breed are still strs, need nums for modeling

# one-hot encode categorical col
df = pd.get_dummies(df, columns=['animal_type', 'sex', 'primary_color', 'primary_breed'])

# check results
print(df.shape)
print(df.dtypes)

# // 97 cols after one-hot encoding is expected, each unique val in categorical col becomes its own binary col

(173710, 97)
adopted                           int64
is_fixed                          int64
has_name                          int64
age_in_days                     float64
outcome_year                      int32
                                 ...   
primary_breed_Other                bool
primary_breed_Pit Bull             bool
primary_breed_Raccoon              bool
primary_breed_Siamese              bool
primary_breed_Siberian Husky       bool
Length: 97, dtype: object


In [36]:
# confirm no missing values
print(df.isnull().sum().sum())

# confirm target var distribution
print(df['adopted'].value_counts(normalize=True).round(3) * 100)

# preview updated df
print(df.shape)

0
adopted
0    50.6
1    49.4
Name: proportion, dtype: float64
(173710, 97)


In [37]:
# save cleaned dataset
df.to_csv("../data/cleaned_adoptions.csv", index=False)

print("Successfully saved")

Successfully saved
